In [14]:
import networkx as nx
import matplotlib.pyplot as plt
import heapq
import numpy as np

class Agent:
    def __init__(self, id, start, goal, speed=1.0):
        self.id = id
        self.current_node = start
        self.goal_node = goal
        self.path = []
        self.position_on_edge = 0.0
        self.speed = speed
        self.time_to_next_node = 0.0
        self.current_edge = None

    def plan_path(self, graph):
        self.path = nx.shortest_path(graph, source=self.current_node, target=self.goal_node, weight='weight')
        self.next_index = 1

    def update(self, graph, dt):
        if self.current_node == self.goal_node:
            return  # Already arrived

        if self.current_edge is None and self.next_index < len(self.path):
            # Start moving along next edge
            u = self.path[self.next_index - 1]
            v = self.path[self.next_index]
            weight = graph[u][v]['weight']
            self.time_to_next_node = weight / self.speed
            self.current_edge = (u, v)
            self.position_on_edge = 0.0

        if self.current_edge:
            self.position_on_edge += dt
            if self.position_on_edge >= self.time_to_next_node:
                # Reached next node
                self.current_node = self.current_edge[1]
                self.current_edge = None
                self.next_index += 1

def create_weighted_graph():

    # erdos renyi graph
    G = nx.erdos_renyi_graph(100, 0.2)
    edges = list(G.edges())
    weight_list = np.random.uniform(1, 5, size=len(edges))  # Genera pesi casuali per gli archi
    for i,e in enumerate(edges):
        u, v = e
        G.add_edge(u, v, weight=weight_list[i] if i < len(weight_list) else 1.0)
    return G, weight_list

def simulate(agents, graph, weight_list, max_time=20, dt=0.5):
    positions = {agent.id: agent.position_on_edge for agent in agents}
    positions_list = [positions.copy()]
    time = 0
    while time < max_time:
        print(f"\nTime: {time:.1f}")
        for agent in agents:
            agent.update(graph, dt)
            print(f"Agent {agent.id} at node {agent.current_node}")
        
        positions = {agent.id: agent.position_on_edge for agent in agents}
        positions_list.append(positions.copy())
        time += dt
    
        # update the weight of the edges based on the agents' positions
        # if the edge has more than a critical number of agents, increase its weight
        for i, curr_edge in enumerate(graph.edges):
            u, v = curr_edge
            agent_count = sum(1 for agent in agents if agent.current_edge == (u, v) or agent.current_edge == (v, u))
            #if agent_count > weight_list[i]:
            #    graph[u][v]['weight'] += 1

            print(f"Edge {u}-{v} has weight {graph[u][v]['weight']} with {agent_count} agents on it")

    return positions_list

# Creazione grafo e agenti
G, weight_list = create_weighted_graph()
agents = [
    Agent(id=i, start=np.random.choice(G.nodes()), goal=np.random.choice(G.nodes())) for i in range(100)
]

# Pianificazione percorsi
for agent in agents:
    agent.plan_path(G)

# Esecuzione simulazione
pos = simulate(agents, G, weight_list)

# (Facoltativo) Disegno del grafo
'''pos = nx.spring_layout(G)
edge_labels = nx.get_edge_attributes(G, 'weight')
nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=500)
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)
plt.title("Grafo pesato")
plt.show()'''


Time: 0.0
Agent 0 at node 2
Agent 1 at node 98
Agent 2 at node 41
Agent 3 at node 88
Agent 4 at node 10
Agent 5 at node 21
Agent 6 at node 57
Agent 7 at node 42
Agent 8 at node 30
Agent 9 at node 13
Agent 10 at node 40
Agent 11 at node 22
Agent 12 at node 63
Agent 13 at node 56
Agent 14 at node 51
Agent 15 at node 79
Agent 16 at node 24
Agent 17 at node 74
Agent 18 at node 37
Agent 19 at node 49
Agent 20 at node 13
Agent 21 at node 13
Agent 22 at node 49
Agent 23 at node 64
Agent 24 at node 63
Agent 25 at node 48
Agent 26 at node 70
Agent 27 at node 50
Agent 28 at node 56
Agent 29 at node 10
Agent 30 at node 42
Agent 31 at node 37
Agent 32 at node 29
Agent 33 at node 68
Agent 34 at node 80
Agent 35 at node 0
Agent 36 at node 66
Agent 37 at node 75
Agent 38 at node 24
Agent 39 at node 20
Agent 40 at node 92
Agent 41 at node 77
Agent 42 at node 57
Agent 43 at node 77
Agent 44 at node 93
Agent 45 at node 21
Agent 46 at node 45
Agent 47 at node 35
Agent 48 at node 7
Agent 49 at node 18
Ag

'pos = nx.spring_layout(G)\nedge_labels = nx.get_edge_attributes(G, \'weight\')\nnx.draw(G, pos, with_labels=True, node_color=\'lightblue\', node_size=500)\nnx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)\nplt.title("Grafo pesato")\nplt.show()'